<a href="https://colab.research.google.com/github/Colin2222-bit/HydrologyML/blob/main/SWE-ML-FirstStep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q cartopy easysnowdata
from pathlib import Path
from google.colab import drive
from google.colab import data_table
import os
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import seaborn as sns
import easysnowdata
import polars as pl
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression

drive.mount('/content/drive', force_remount=True)
PROJECT_DIR = Path("/content/drive/MyDrive/SWE_Project")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
archive_path = PROJECT_DIR / "snotel_data.nc"


if archive_path.exists():
    print("Loading cached SNOTEL archive from Drive...")
    ds = xr.open_dataset(archive_path)

else:
    print("Archive not found on Drive. Downloading...")


    AllStations = easysnowdata.automatic_weather_stations.StationCollection()
    AllStations.get_entire_data_archive()
    ds = AllStations.entire_data_archive


    if 'geometry' in ds.variables:
        ds = ds.drop_vars('geometry')


    print("Saving archive to Google Drive...")
    ds.to_netcdf(archive_path)
    print("Saved successfully!")

print('Done!')
ds=ds.where(ds['WY'] > 2000)




Mounted at /content/drive
Loading cached SNOTEL archive from Drive...
Done!


In [ ]:

for var in ["WTEQ", "PRCPSA", "SNWD", "TMAX", "TMIN", "TAVG"]:

        ds[var] = ds[var].astype("float32")
def missing_days (ds):

    # 1. Identify missing data within the first 15 days
    swe_missing_early = (ds["DOWY"] <= 15) & ds["WTEQ"].isnull()
    prcp_missing_early = (ds["DOWY"] <= 15) & ds["PRCPSA"].isnull()

    # 2. Find any Water Year that has this problem
    either = swe_missing_early | prcp_missing_early
    bad_wy = either.groupby('WY').sum(dim='time') >= 3

    # 4. Expand the mask to all days in those bad years
    bad_wy_mask = bad_wy.sel(WY=ds['WY'])

    # 5. Drop the bad data. Do NOT save 'either' as a flag!
    early_bad = bad_wy_mask & (ds['DOWY'] <= 15)   # or <= 30
    total_bad_years_prop = 100 * int(early_bad.sum().values) / early_bad.size

    ds = ds.where(~early_bad)


    return ds

def apply_physical_bounds_qc(ds):
    """
    Module 1: Removes values that break the laws of physics or regional climate.
    Uses Serreze for Temp/Precip, Durre for SNWD.
    """
    # 1. Negative Sweeps (Unified)
    for var in ['WTEQ', 'PRCPSA', 'SNWD']:
       ds[f'FLAG_{var}_NEGATIVE'] = ds[var] < 0
       ds[var] = ds[var].where(ds[var] >= 0)


    # 2. Extreme Precipitation (Serreze: > 254 mm/day)
    ds['FLAG_PRCP_EXTREME'] = ds['PRCPSA'] > 0.254
    ds['PRCPSA'] = ds['PRCPSA'].where(~ds['FLAG_PRCP_EXTREME'])

    # 3. Extreme Snow Depth (Durre: > 11.46 m)
    if 'SNWD' in ds:
        ds['FLAG_SNWD_EXTREME'] = ds['SNWD'] > 11.46
        ds['SNWD'] = ds['SNWD'].where(~ds['FLAG_SNWD_EXTREME'])

    # 4. Temperature Bounds (Serreze with Alaska Mod)
    alaska_mask = ds['state'] == 'Alaska'
    temp_extreme_any = xr.zeros_like(ds["TAVG"], dtype=bool)

    for temp in ['TMAX', 'TMIN', 'TAVG']:
        flag = (ds[temp] > 40) | \
               ((ds[temp] < -40) & ~alaska_mask) | \
               ((ds[temp] < -60) & alaska_mask)
        ds[f'FLAG_{temp}_EXTREME'] = flag
        temp_extreme_any = temp_extreme_any | flag # If one breaks, they all break

    # Apply the mask to all temperature variables simultaneously
    for temp in ['TMAX', 'TMIN', 'TAVG']:
        ds[temp] = ds[temp].where(~temp_extreme_any)

    return ds

def apply_hardware_spike_qc(ds):
    """
    Module 2: Identifies broken or stuck sensors using Durre's methods.
    """
    # 1. Temperature Streaks (20 days)
    for var in ['TMAX', 'TMIN', 'TAVG']:
        # min_periods=20 ensures it only flags if 20 valid days exist
        ds[f'FLAG_{var}_STREAK'] = ds[var].rolling(time=20, min_periods=20).std() < 1e-5
        flag = ds[f'FLAG_{var}_STREAK']
        expanded = flag.rolling(time=20, min_periods=1).max().astype(bool)
        ds[f'FLAG_{var}_STREAK_EXPANDED'] = expanded
        ds[var] = ds[var].where(~expanded)

    # 2. Temperature Spikes/Dips (Isolated 25C jumps)
    for var in ['TMAX', 'TMIN']:
        diff_yesterday = ds[var] - ds[var].shift(time=1)
        diff_tomorrow = ds[var] - ds[var].shift(time=-1)

        spike = (diff_yesterday >= 25) & (diff_tomorrow >= 25)
        dip = (diff_yesterday <= -25) & (diff_tomorrow <= -25)
        ds[f'FLAG_{var}_SPIKE_DIP'] = spike | dip

        # Mask the spikes/dips
        ds[var] = ds[var].where(~ds[f'FLAG_{var}_SPIKE_DIP'])

    # 3. SNWD Stagnation (90 days of NON-ZERO snow)

    snwd_nonzero = ds['SNWD'].where(ds['SNWD'] > 0)

    # Get the strict 90-day flag
    stagnant = snwd_nonzero.rolling(time=90, min_periods=90).std() == 0
    ds['FLAG_SNWD_STAGNANT'] = stagnant

        # Expand it backwards by 90 days to mask the WHOLE stuck period (not just 20)
    expanded = stagnant.rolling(time=90, min_periods=1).max().astype(bool)
    ds['FLAG_SNWD_STAGNANT_EXPANDED'] = expanded

        # Explicitly mask SNWD (No 'var' variable used!)
    ds['SNWD'] = ds['SNWD'].where(~expanded)
    return ds

def apply_snow_physics_qc(ds):
    """
    Consolidated Module 3: Checks daily changes in SWE and SNWD.
    Includes Density Physics, Serreze Gross Jumps, Reversals, and Durre Warm Inc.
    """
    # 1. Calculate daily changes
    swe_diff = ds['WTEQ'] - ds['WTEQ'].shift(time=1)
    snwd_diff = ds['SNWD'] - ds['SNWD'].shift(time=1)

    # 2. Advanced Density Physics Checks
    # Calculate Density safely
    density = ds['WTEQ'] / ds['SNWD'].where(ds['SNWD'] > 0)
    density_diff = density - density.shift(time=1)

    # Ignore Day 1 of Water Year
    valid_days = ds['DOWY'] != 1

    # Rule 1: Neg SWE, Neg Density, but POSITIVE Depth (Impossible fluffing)
    rule1 = (swe_diff < 0) & (density_diff < 0) & (snwd_diff > 0)
    # Rule 2: Pos SWE, Neg Depth, but RAPID Density spike (Impossible compaction)
    rule2 = (swe_diff > 0) & (snwd_diff < 0) & (density_diff > 0.0005)

    ds['FLAG_PHYSICS_QC_FAIL'] = (rule1 | rule2) & valid_days

    # Mask data
    ds['WTEQ'] = ds['WTEQ'].where(~ds['FLAG_PHYSICS_QC_FAIL'])
    ds['SNWD'] = ds['SNWD'].where(~ds['FLAG_PHYSICS_QC_FAIL'])

    # 3. Serreze Gross SWE Jumps
    is_gross_jump = abs(swe_diff.where(valid_days)) > 0.254
    is_sensor_malfunction = is_gross_jump & (ds['PRCPSA'] < 0.200)
    ds['FLAG_SWE_GROSS'] = is_sensor_malfunction
    ds['WTEQ'] = ds['WTEQ'].where(~is_sensor_malfunction)

    # 4. Serreze Reversals
    swe_diff_tomorrow = swe_diff.shift(time=-1)
    swe_jump = ((swe_diff > 0.0635) & (swe_diff_tomorrow < -0.0635)) | \
               ((swe_diff < -0.0635) & (swe_diff_tomorrow > 0.0635))
    ds['FLAG_SWE_REVERSAL'] = (swe_jump | swe_jump.shift(time=1, fill_value=False)) & valid_days
    ds['WTEQ'] = ds['WTEQ'].where(~(ds['FLAG_SWE_GROSS'] | ds['FLAG_SWE_REVERSAL']))

    # 5. Durre Warm SNWD Increase
    if 'SNWD' in ds:
        tmin_prev = ds["TMIN"].shift(time=1)
        tmin_curr = ds["TMIN"]
        tmin_next = ds["TMIN"].shift(time=-1)
        tmin_3day = xr.apply_ufunc(np.fmin, xr.apply_ufunc(np.fmin, tmin_prev, tmin_curr), tmin_next)

        ds['FLAG_SNWD_WARM_INC'] = (snwd_diff > 0) & (tmin_3day >= 7) & valid_days
        ds['FLAG_SNWD_SPIKE'] = (snwd_diff > 1.925) & valid_days
        ds['SNWD'] = ds['SNWD'].where(~(ds['FLAG_SNWD_SPIKE'] | ds['FLAG_SNWD_WARM_INC']))

    return ds


def apply_snotel_bc(ds):
  """Module 4: Applies Harms et al. temperature correction to SNOTEL only."""
  is_snotel = ds['network'] == 'SNOTEL'
  ds['is_snotel'] = is_snotel
  for var in ['TMAX', 'TMIN', 'TAVG']:
       corrected_temp = (1.03 * ds[var]) - 0.9
       ds[var] = xr.where(is_snotel, corrected_temp, ds[var])
  return ds


#Apply all the basic checks/bc FIRST (no leakage issue)


def split_ds(ds):
    # Make sure to RETURN the datasets so you can use them in the next function!
    training = ds.where((ds["WY"] >= 2001) & (ds["WY"] <= 2015), drop=True)
    validation = ds.where((ds["WY"] >= 2016) & (ds["WY"] <= 2019), drop=True)
    testing = ds.where((ds["WY"] >= 2020) & (ds["WY"] <= 2025), drop=True)

    return training, validation, testing




def gap_fill_predictors(ds):
    # 1. Add the missing loop for temperature variables
    for var in ['TMAX', 'TMIN', 'TAVG']:
        was_null = ds[var].isnull()
        ds[var] = ds[var].interpolate_na(dim='time', method='linear', limit=2)
        ds[f'FLAG_{var}_INTERP'] = was_null & ds[var].notnull()


    # 2. SNWD Logic
    was_missing = ds['SNWD'].isnull()
    ds['SNWD'] = ds['SNWD'].interpolate_na(dim='time', method='linear', limit=7)
    ds['FLAG_SNWD_FILLED'] = was_missing & ds['SNWD'].notnull()

    # 3. PRCPSA Logic (Moved UP so it actually runs before the return)
    prcp_nulls = ds['PRCPSA'].isnull()
    is_gap_end = prcp_nulls.rolling(time=4, min_periods=4).sum() == 4
    large_prcp_gap = (is_gap_end | is_gap_end.shift(time=-1, fill_value=False) |
                      is_gap_end.shift(time=-2, fill_value=False) |
                      is_gap_end.shift(time=-3, fill_value=False)) & prcp_nulls

    fill_mask = prcp_nulls & ~large_prcp_gap
    ds['PRCPSA'] = ds['PRCPSA'].fillna(0)

    ds['FLAG_PRCP_FILLED'] = fill_mask
    ds['PRCPSA'] = ds['PRCPSA'].where(~large_prcp_gap)


    return ds

print(f"1. Raw archive stations: {len(np.unique(ds['station']))}")






# 5. After missing_days (This is often the 'Station Killer')



ds = apply_snotel_bc(ds)              # fixed coeffs — fine pre-split
ds = apply_physical_bounds_qc(ds)
print(f"3. After Physical Bounds QC: {len(np.unique(ds['station']))}") # fixed — fine pre-split
ds = apply_hardware_spike_qc(ds)      # fixed + local rolling — fine, avoids edge artifacts
ds = apply_snow_physics_qc(ds)

print(f"4. After Snow Physics QC: {len(np.unique(ds['station']))}")      # fixed — fine pre-split
ds = missing_days(ds)                 # fixed count — fine pre-split

print(f"5. After Missing Days QC: {len(np.unique(ds['station']))}")

training, validation, testing = split_ds(ds)


training   = gap_fill_predictors(training)
validation = gap_fill_predictors(validation)
testing    = gap_fill_predictors(testing)


def toparquet (ds, feature_cols, target_col='WTEQ'):

    keep = feature_cols + [target_col]

    small = ds[keep]

    for v in small.data_vars:
        if "FLAG" in v or v.startswith("NO"):
            small[v] = small[v].fillna(False).astype("uint8")
        else:
            small[v] = small[v].astype("float32")

    df = small.to_dataframe().reset_index()
    df = df.dropna(subset=keep)

    return df
feature_cols = [
      'PRCPSA', 'TMAX', 'TMIN', 'DOWY', 'TAVG', 'SNWD',
        'FLAG_PRCP_FILLED', 'FLAG_TMAX_INTERP',
        'FLAG_TMIN_INTERP',  'FLAG_TAVG_INTERP', 'FLAG_SNWD_FILLED', 'is_snotel'
    ]
train_df = toparquet(training, feature_cols, 'WTEQ')

train_df.to_parquet(PROJECT_DIR / "snotel_training1.parquet")
del train_df

val_df = toparquet(validation, feature_cols, 'WTEQ')
val_df.to_parquet(PROJECT_DIR / "snotel_validation1.parquet")
del val_df

test_df = toparquet(testing, feature_cols, 'WTEQ')
test_df.to_parquet(PROJECT_DIR / "snotel_testing1.parquet")
del test_df


1. Raw archive stations: 969
3. After Physical Bounds QC: 969
4. After Snow Physics QC: 969
5. After Missing Days QC: 969


In [ ]:

fig, ax = plt.subplots(1, 6, figsize=(26, 5))
colors = {
    "SNWD": "#1f77b4",    # Deep Blue for Snow Depth
    "WTEQ": "#aec7e8",    # Light Blue for Snow Water Equivalent (SWE)
    "PRCPSA": "#ff7f0e"   # Orange for Accumulated Precipitation
}
# --- Row 1 / Subplot 0-2: Monthly Averages ---
# --- Row 1 / Subplot 0-2: Monthly Averages ---
# Store the grouped data to keep your code clean
snwd_monthly = ds.SNWD.groupby(ds.time.dt.month).mean(dim=['time', 'station'])
wteq_monthly = ds.WTEQ.groupby(ds.time.dt.month).mean(dim=['time', 'station'])
prcpsa_monthly = ds.PRCPSA.groupby(ds.time.dt.month).mean(dim=['time', 'station'])

# Pass the '.month' coordinate explicitly as the X-axis value
ax[0].plot(snwd_monthly.month, snwd_monthly, color=colors["SNWD"], linewidth=2.5)
ax[0].set_title("Monthly Mean Snow Depth", fontsize=11, fontweight='bold')
ax[0].set_ylabel("Snow Depth (m)", fontsize=10)

ax[1].plot(wteq_monthly.month, wteq_monthly, color=colors["WTEQ"], linewidth=2.5)
ax[1].set_title("Monthly Mean SWE", fontsize=11, fontweight='bold')
ax[1].set_ylabel("SWE (m)", fontsize=10)

ax[2].plot(prcpsa_monthly.month, prcpsa_monthly, color=colors["PRCPSA"], linewidth=2.5)
ax[2].set_title("Monthly Mean Accum. Precip", fontsize=11, fontweight='bold')
ax[2].set_ylabel("Precipitation (m)", fontsize=10)

# Apply standard monthly formatting to the first 3 plots
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
for i in range(3):
    ax[i].set_xlabel("Month", fontsize=10)
    ax[i].set_xticks(range(1, 13))
    ax[i].set_xticklabels(month_labels, rotation=45)
    # FIX 1: Lock the boundaries precisely onto your data range (1 to 12)
    ax[i].set_xlim(1,12)
    ax[i].grid(True, linestyle="--", alpha=0.5)

# --- Row 1 / Subplot 3-5: Day of Year (Daily Climatology) ---
# FIX 2: Exclude day 366 from the calculated data arrays using .sel()
snwd_daily = ds.SNWD.groupby(ds.time.dt.dayofyear).mean(dim=['time', 'station'])
wteq_daily = ds.WTEQ.groupby(ds.time.dt.dayofyear).mean(dim=['time', 'station'])
prcpsa_daily = ds.PRCPSA.groupby(ds.time.dt.dayofyear).mean(dim=['time', 'station'])

# Safely drop day 366 if it exists in the grouped coordinate index
days_to_keep = [d for d in snwd_daily.dayofyear.values if d <= 365]
snwd_daily = snwd_daily.sel(dayofyear=days_to_keep)
wteq_daily = wteq_daily.sel(dayofyear=days_to_keep)
prcpsa_daily = prcpsa_daily.sel(dayofyear=days_to_keep)

ax[3].plot(snwd_daily.dayofyear, snwd_daily, color=colors["SNWD"], alpha=0.8)
ax[3].set_title("Daily Climatology: Snow Depth", fontsize=11, fontweight='bold')

ax[4].plot(wteq_daily.dayofyear, wteq_daily, color=colors["WTEQ"], alpha=0.8)
ax[4].set_title("Daily Climatology: SWE", fontsize=11, fontweight='bold')

ax[5].plot(prcpsa_daily.dayofyear, prcpsa_daily, color=colors["PRCPSA"], alpha=0.8)
ax[5].set_title("Daily Climatology: Accum. Precip", fontsize=11, fontweight='bold')

# Apply daily formatting to the last 3 plots
for i in range(3, 6):
    ax[i].set_xlabel("Day of Year (1-365)", fontsize=10)

    ax[i].set_xlim(1, 365)
    ax[i].grid(True, linestyle="--", alpha=0.5)

fig.suptitle("SNOTEL Seasonal Climatology Profiles", fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()


In [ ]:


# 1. Map boundaries based on your total dataset with a 2-degree padding
lon_min, lon_max = float(ds.longitude.min()) - 2, float(ds.longitude.max()) + 2
lat_min, lat_max = float(ds.latitude.min()) - 2, float(ds.latitude.max()) + 2


# 3. Create a 2x3 grid of subplots with PlateCarree projection
fig, axes = plt.subplots(2, 3, figsize=(22, 14), subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()

# Helper function to apply map features
def style_snotel_map(ax, title):
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='whitesmoke')
    ax.add_feature(cfeature.OCEAN, facecolor='aliceblue')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black', linewidth=0.8)
    ax.add_feature(cfeature.STATES, linestyle='--', edgecolor='gray', linewidth=0.5)

    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, linestyle=':', color='gainsboro')
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(title, fontsize=13, weight='bold', pad=10)

# Map configurations
vars_to_plot = ['TMAX', 'TMIN', 'TAVG', 'SNWD', 'WTEQ', 'PRCPSA']
cmaps = ['YlOrRd', 'coolwarm', 'RdYlBu_r', 'Blues', 'YlGnBu', 'viridis']
labels = ['Mean Max Temp (°C)', 'Mean Min Temp (°C)', 'Mean Temp (°C)', 'Mean Snow Depth (m)', 'Mean Water Equiv. (m)', 'Mean Precip Acc. (m)']

# Populate the SNOTEL grid
for i, var in enumerate(vars_to_plot):
    style_snotel_map(axes[i], f'SNOTEL Stations\n{var} (All-Time Mean)')

    spatial_data = ds[var].mean(dim='time', skipna=True)

    sc = axes[i].scatter(
        ds.longitude, _ds.latitude,
        c=spatial_data,
        transform=ccrs.PlateCarree(),
        cmap=cmaps[i], edgecolors='black', linewidth=0.5, s=55, zorder=3
    )
    cbar = fig.colorbar(sc, ax=axes[i], orientation='horizontal', pad=0.08, shrink=0.8, aspect=25)
    cbar.set_label(labels[i], fontsize=10)

fig.suptitle("SNOTEL Network Regional Spatial Analysis", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()
